In [125]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.preprocessing import StandardScaler

In [126]:
X=pd.read_csv("train.csv")


In [127]:
test_df

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...
413,1305,3,"Spector, Mr. Woolf",male,NaN,0,0,A.5. 3236,8.0500,NaN,S
414,1306,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C
415,1307,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S
416,1308,3,"Ware, Mr. Frederick",male,NaN,0,0,359309,8.0500,NaN,S


In [128]:


X = df.drop(columns=["PassengerId", "Survived", "Name", "Ticket", "Cabin"])

y = df["Survived"]
X["Sex"] = X["Sex"].map({"female": 1, "male": 0})
X["Embarked"] = X["Embarked"].map({'S':1, 'C':2, 'Q':3,})
X["Sex_Fare"] = X["Sex"] * X["Fare"]
X["Sex_Age"] = X["Sex"] * X["Age"]

X = X.fillna(0)

scaler = StandardScaler()
X_scaled_array = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled_array, columns=X.columns)



print(X_scaled.head())

     Pclass       Sex       Age     SibSp     Parch      Fare  Embarked  \
0  0.827377 -0.737695 -0.102313  0.432793 -0.473674 -0.502445 -0.562619   
1 -1.566107  1.355574  0.807492  0.432793 -0.473674  0.786845  1.003923   
2  0.827377  1.355574  0.125138 -0.474545 -0.473674 -0.488854 -0.562619   
3 -1.566107  1.355574  0.636903  0.432793 -0.473674  0.420730 -0.562619   
4  0.827377 -0.737695  0.636903 -0.474545 -0.473674 -0.486337 -0.562619   

   Sex_Fare   Sex_Age  
0 -0.387882 -0.551937  
1  1.376012  2.012910  
2 -0.191779  1.202959  
3  0.926069  1.810422  
4 -0.387882 -0.551937  


In [61]:
X["Embarked"].unique()

array([ 1.,  2.,  3., nan])

In [129]:
# lr=LogisticRegression()
# dt=DecisionTreeClassifier(max_depth=3)
# rf=RandomForestClassifier(n_estimators=201)
# voting_clf=VotingClassifier(
#     estimators=[
#         ("lr",lr),
#         ("dt",dt),
#         ("rf",rf)
    
# ])
# voting_clf.fit(X,y)


lr = LogisticRegression(max_iter=1000) 
dt = DecisionTreeClassifier(max_depth=3)
rf = RandomForestClassifier(n_estimators=201)

# 2. Define the Voting Classifier
voting_clf = VotingClassifier(
    estimators=[
        ("lr", lr),
        ("dt", dt),
        ("rf", rf)
    ],
    voting='soft' # Changed to 'soft' for better performance
)

# 3. Fit the model
voting_clf.fit(X_scaled, y)

# 4. Check accuracy (Optional)
# print(voting_clf.score(X, y))

,estimators,"[('lr', ...), ('dt', ...), ...]"
,voting,'soft'
,weights,None
,n_jobs,None
,flatten_transform,True
,verbose,False
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True


In [131]:

X_test = pd.read_csv("test.csv")
passenger_ids = X_test["PassengerId"]
# Test cell — step 3
X_test = X_test.drop(columns=["PassengerId","Name", "Ticket", "Cabin"])
X_test["Sex"] = X_test["Sex"].map({"female": 1, "male": 0})
X_test["Embarked"] = X_test["Embarked"].map({'S': 1, 'C': 2, 'Q': 3})

X_test["Sex_Fare"] = X_test["Sex"] * X_test["Fare"]
X_test["Sex_Age"] = X_test["Sex"] * X_test["Age"]

X_test = X_test.fillna(0)

X_test_scaled_array = scaler.transform(X_test)
X_test_scaled = pd.DataFrame(X_test_scaled_array, columns=X_test.columns)
                             
y_pred_kaggle = voting_clf.predict(X_test_scaled)
submission = pd.DataFrame({"PassengerId": passenger_ids, "Survived": y_pred_kaggle})
submission.to_csv("submission.csv", index=False)
print("submission.csv successfully created!")

submission.csv successfully created!


,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...
413,1305,3,"Spector, Mr. Woolf",male,NaN,0,0,A.5. 3236,8.0500,NaN,S
414,1306,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C
415,1307,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S
416,1308,3,"Ware, Mr. Frederick",male,NaN,0,0,359309,8.0500,NaN,S
